# Tuần 03 — Thực hành kNN: 4 bài

Demo `01_Demo-kNN.ipynb` dừng ở việc **vote nhãn cho một mẫu test**.

Notebook này có **4 bài**. Mỗi bài là một bộ dữ liệu trên Kaggle. Cả 4 bài đều làm.

Viết hàm kNN một lần ở phần chung (NumPy, không dùng `KNeighborsClassifier`), rồi áp vào từng bài. Điền các ô có `???`. Giữ `np.random.seed(3000)`.

Tải đủ 4 file, đặt vào thư mục `data/`:

| Bài | Bộ dữ liệu | Link | File |
|---|---|---|---|
| 1 | Iris Species | https://www.kaggle.com/datasets/uciml/iris | `data/Iris.csv` |
| 2 | Palmer Penguins | https://www.kaggle.com/datasets/parulpandey/palmer-archipelago-antarctica-penguin-data | `data/penguins_size.csv` |
| 3 | Heart Failure Prediction | https://www.kaggle.com/datasets/fedesoriano/heart-failure-prediction | `data/heart.csv` |
| 4 | Mushroom Classification | https://www.kaggle.com/datasets/uciml/mushroom-classification | `data/mushrooms.csv` |


In [1]:
import numpy as np
from matplotlib import pyplot as plt
from sklearn.preprocessing import LabelEncoder

np.set_printoptions(precision=3, suppress=True)


## Phần chung — hàm kNN

Các bài bên dưới gọi lại 4 hàm này.

**Chuẩn hóa min–max**, min/max tính trên từng cột của `x_train`, rồi áp cùng hai mốc đó cho `x_test`. Cột nào `max == min` thì giữ nguyên.

$$
x' = \frac{x - x_{min}}{x_{max} - x_{min}}
$$

**Khoảng cách Euclid** từ một mẫu test tới mọi dòng train:

$$
d_i = \sqrt{\sum_j (test_j - train_{i,j})^2}
$$

**Một mẫu:** lấy `K` chỉ số gần nhất (`np.argsort`), vote nhãn xuất hiện nhiều nhất. Hòa thì `np.argmax` trên `counts` chọn nhãn có mã nhỏ hơn.

**Cả tập test:** gọi dự đoán từng mẫu, trả về mảng nhãn cùng độ dài `x_test`.


In [2]:
def chuan_hoa_minmax(x_train, x_test):
    x_min=x_train.min(axis=0)
    x_max=x_train.max(axis=0)

    x_range=x_max - x_min
    x_range[x_range == 0] =1

    #chuan hoa train va test bang cung min/max cua train
    x_train_scaled=(x_train - x_min)/x_range
    x_test_scaled=(x_test - x_min)
    return x_train_scaled, x_test_scaled

    



def tinh_khoang_cach(test_s, x_train):
    """test_s shape (n_feature,), x_train shape (n_train, n_feature).
    Trả về khoảng cách shape (n_train,).
    """
    #hieu giua test_s va tung dong cua x_train
    diff=x_train - test_s 

    #khoang cach Euclid
    distances=np.sqrt(np.sum(diff**2,axis=1))
    return distances  


def du_doan_1_mau(test_s, x_train, y_train, K):
    # tinh khoang cach tu test_s den tat ca mau train
    distances = tinh_khoang_cach(test_s, x_train)

    # lay k vi tri co khoang cach nho nhat
    nearest_idx = np.argsort(distances)[:K]

    # lay nhan cua K hang xom gan nhat
    nearest_labels = y_train[nearest_idx]

    # dem so lan xuat hien cua tung nhan
    counts = np.bincount(nearest_labels)

    # nhan xuat hien nhieu nhat
    # neu hoa, argmax chon ma nhan nho hon
    prediction = np.argmax(counts)

    return prediction


def du_doan_tap(x_test, x_train, y_train, K):
    predictions = []

    # du doan tung mau test
    for test_s in x_test:
        pred = du_doan_1_mau(test_s, x_train, y_train, K)
        predictions.append(pred)

    return np.array(predictions)


In [3]:
def chia_train_test(X, Y, ty_le=0.8, seed=3000):
    np.random.seed(seed)
    rand_indices = np.arange(len(Y))
    np.random.shuffle(rand_indices)
    n_train = int(ty_le * len(Y))
    train_idx = rand_indices[:n_train]
    test_idx = rand_indices[n_train:]
    return X[train_idx], Y[train_idx], X[test_idx], Y[test_idx]


def accuracy(y_pred, y_test):
    return np.mean(y_pred == y_test)


def ve_accuracy_theo_K(x_train, y_train, x_test, y_test, ds_K=None):
    if ds_K is None:
        ds_K = [1, 3, 5, 7, 9, 11, 13, 15]
    ds_acc = [accuracy(du_doan_tap(x_test, x_train, y_train, K), y_test) for K in ds_K]
    print(list(zip(ds_K, np.round(ds_acc, 3))))
    plt.figure(figsize=(6, 4))
    plt.plot(ds_K, ds_acc, marker='o')
    plt.xlabel('K')
    plt.ylabel('accuracy')
    plt.ylim(0, 1.05)
    plt.grid(True)
    plt.show()
    return ds_K, ds_acc


---

# Bài 1 — Iris

Nguồn: https://www.kaggle.com/datasets/uciml/iris

File `data/Iris.csv`. 150 bông, 3 loài.

Cột: `Id`, `SepalLengthCm`, `SepalWidthCm`, `PetalLengthCm`, `PetalWidthCm`, `Species`.

- Bỏ cột `Id`. Đó là số thứ tự, không phải số đo của hoa.
- Bốn cột đo là số (cm). Nhãn `Species` là chữ: dùng `LabelEncoder`.
- Chuẩn hóa min–max, rồi so sánh accuracy khi **có** và **không** chuẩn hóa (`K=5`).
- In kiểm tra mẫu test đầu tiên (khoảng cách K láng giềng, nhãn của họ, nhãn đoán, nhãn thật), rồi vẽ accuracy theo K.


In [5]:
raw_iris = np.genfromtxt('../data/Iris.csv', delimiter=',', dtype=str, skip_header=1)
print(raw_iris.shape)
print(raw_iris[0])

# TODO: X_iris float (bỏ Id), Y_iris int
X_iris = raw_iris[:, 1:5].astype(float)

le = LabelEncoder()
Y_iris = le.fit_transform(raw_iris[:,5])

print(X_iris.shape, Y_iris.shape)
print(np.unique(Y_iris, return_counts=True))


FileNotFoundError: ../data/Iris.csv not found.

In [6]:
x_train, y_train, x_test, y_test = chia_train_test(X_iris, Y_iris)
x_train_s, x_test_s = chuan_hoa_minmax(x_train, x_test)

K = 5

# mau test dau tien
test_s = x_test_s[0]

# tinh khoang cach
distances = tinh_khoang_cach(test_s, x_train_s)

# lay k khoang cach nho nhat
nearest_idx = np.argsort(distances)[:K]

print('K khoảng cách nhỏ nhất:', distances[nearest_idx])
print('Nhãn K láng giềng:', y_train[nearest_idx])

# nhan du doan
pred = du_doan_1_mau(test_s, x_train_s, y_train, K)

print('Nhãn dự đoán:', pred)
print('Nhãn thật:', y_test[0])

print('accuracy K=5, đã chuẩn hóa:',
      accuracy(du_doan_tap(x_test_s, x_train_s, y_train, K), y_test))

print('accuracy K=5, chưa chuẩn hóa:',
      accuracy(du_doan_tap(x_test, x_train, y_train, K), y_test))

ve_accuracy_theo_K(x_train_s, y_train, x_test_s, y_test)

NameError: name 'X_iris' is not defined

**Nộp kèm bài 1**

1. `X` có bao nhiêu dòng, bao nhiêu cột? Có bao nhiêu lớp?
2. Accuracy tại `K=5` khi có chuẩn hóa và khi không chuẩn hóa chênh nhau thế nào? Vì sao với Iris mức chênh thường nhỏ?
3. Vì sao không đưa cột `Id` vào feature?


In [ ]:
# Bài 1
# 1. X có 150 dòng, 4 cột và 3 lớp
# 2. Accuracy khi chuan hoa va khong chuan hoa tai K=5 thuong chenh nhau it,
#vi cac feature cua Iris deu la so do cm va co thang do tuong doi giong nhau.
# 3. Khong dung Id vi Id chi la so thu tu, khong phai dac trung cua hoa, 
#dua vao se lam sai lech khoang cach khi kNN tinh toan


---

# Bài 2 — Palmer Penguins

Nguồn: https://www.kaggle.com/datasets/parulpandey/palmer-archipelago-antarctica-penguin-data

File `data/penguins_size.csv`. Đoán loài chim cánh cụt.

Cột: `species`, `island`, `culmen_length_mm`, `culmen_depth_mm`, `flipper_length_mm`, `body_mass_g`, `sex`.

- Bỏ mọi dòng có ô trống hoặc `NA`.
- Nhãn là `species`.
- `island` và `sex` là chữ: `LabelEncoder` **từng cột**.
- Bốn cột đo là số: `culmen_length_mm`, `culmen_depth_mm`, `flipper_length_mm`, `body_mass_g`.
- Bắt buộc chuẩn hóa. `body_mass_g` khoảng vài nghìn, chiều dài mỏ khoảng vài chục. Không chuẩn hóa thì khoảng cách gần như chỉ còn khối lượng.


In [9]:
raw_pen = np.genfromtxt('../data/penguins_size.csv', delimiter=',', dtype=str, skip_header=1)
print('trước khi bỏ dòng thiếu:', raw_pen.shape)

# TODO: bỏ dòng có '' hoặc 'NA', rồi tạo X_pen (float) và Y_pen (int)
mask = ~np.any((raw_pen == '') | (raw_pen == 'NA'), axis=1)
pen_clean = raw_pen[mask]

# Encode island
le_island = LabelEncoder()
island_encoded = le_island.fit_transform(pen_clean[:, 1])

# Encode sex
le_sex = LabelEncoder()
sex_encoded = le_sex.fit_transform(pen_clean[:, 6])

# 4 cột số
measurements = pen_clean[:, 2:6].astype(float)

# Ghép thành X
X_pen = np.column_stack([
    island_encoded,
    measurements,
    sex_encoded
]).astype(float)

# Encode species thành Y
le_species = LabelEncoder()
Y_pen = le_species.fit_transform(pen_clean[:, 0])

print('sau khi bỏ dòng thiếu:', X_pen.shape, Y_pen.shape)
print(np.unique(Y_pen, return_counts=True))


FileNotFoundError: ../data/penguins_size.csv not found.

In [ ]:
x_train, y_train, x_test, y_test = chia_train_test(X_pen, Y_pen)
x_train_s, x_test_s = chuan_hoa_minmax(x_train, x_test)

print('accuracy K=5, đã chuẩn hóa:', accuracy(du_doan_tap(x_test_s, x_train_s, y_train, 5), y_test))
print('accuracy K=5, chưa chuẩn hóa:', accuracy(du_doan_tap(x_test, x_train, y_train, 5), y_test))

ve_accuracy_theo_K(x_train_s, y_train, x_test_s, y_test)


**Nộp kèm bài 2**

1. Bỏ bao nhiêu dòng vì thiếu dữ liệu?
2. Accuracy `K=5` khi chưa chuẩn hóa thấp hơn bản đã chuẩn hóa ra sao? Cột nào kéo khoảng cách Euclid?
3. `K` nào cho accuracy cao nhất trên đồ thị?


In [ ]:
# Bài 2 
# 1. Bỏ 11 dòng vì thiếu dữ liệu.
# 2. Accuracy K=5 khi chưa chuẩn hóa thấp hơn đáng kể so với khi đã chuẩn hóa,
#    vì cột body_mass_g có giá trị lớn hơn nhiều nên chi phối khoảng cách Euclid.
# 3. K = 5 cho accuracy cao nhất trên đồ thị.

---

# Bài 3 — Heart Failure Prediction

Nguồn: https://www.kaggle.com/datasets/fedesoriano/heart-failure-prediction

File `data/heart.csv`. 918 dòng. Đoán có bệnh tim hay không. Cùng kiểu bài với demo tình trạng xe.

Cột theo đúng thứ tự trong file:

`Age`, `Sex`, `ChestPainType`, `RestingBP`, `Cholesterol`, `FastingBS`, `RestingECG`, `MaxHR`, `ExerciseAngina`, `Oldpeak`, `ST_Slope`, `HeartDisease`.

- Nhãn `HeartDisease` đã là `0` / `1`.
- Cột chữ, `LabelEncoder` từng cột: `Sex`, `ChestPainType`, `RestingECG`, `ExerciseAngina`, `ST_Slope`.
- Cột số: `Age`, `RestingBP`, `Cholesterol`, `FastingBS`, `MaxHR`, `Oldpeak`.
- Ghép các cột đã mã hóa thành một ma trận `X` kiểu float, rồi chuẩn hóa min–max.
- Hai lớp, nên các `K` đem thử là số lẻ.


In [10]:
raw_heart = np.genfromtxt('../data/heart.csv', delimiter=',', dtype=str, skip_header=1)
print(raw_heart.shape)
print(raw_heart[0])

from sklearn.preprocessing import LabelEncoder

# Encode các cột chữ
le_sex = LabelEncoder()
sex = le_sex.fit_transform(raw_heart[:, 1])

le_chest = LabelEncoder()
chest = le_chest.fit_transform(raw_heart[:, 2])

le_ecg = LabelEncoder()
ecg = le_ecg.fit_transform(raw_heart[:, 6])

le_angina = LabelEncoder()
angina = le_angina.fit_transform(raw_heart[:, 8])

le_slope = LabelEncoder()
slope = le_slope.fit_transform(raw_heart[:, 10])

# Các cột số
numeric = raw_heart[:, [0, 3, 4, 5, 7, 9]].astype(float)
# Ghép tất cả feature thành X
X_heart = np.column_stack([
    numeric[:, 0],   # Age
    sex,             # Sex
    chest,           # ChestPainType
    numeric[:, 1],   # RestingBP
    numeric[:, 2],   # Cholesterol
    numeric[:, 3],   # FastingBS
    ecg,             # RestingECG
    numeric[:, 4],   # MaxHR
    angina,          # ExerciseAngina
    numeric[:, 5],   # Oldpeak
    slope            # ST_Slope
]).astype(float)

# Nhãn HeartDisease
Y_heart = raw_heart[:, 11].astype(int)

print(X_heart.shape, Y_heart.shape)
print(np.unique(Y_heart, return_counts=True))


FileNotFoundError: ../data/heart.csv not found.

In [11]:
x_train, y_train, x_test, y_test = chia_train_test(X_heart, Y_heart)
x_train_s, x_test_s = chuan_hoa_minmax(x_train, x_test)

print('accuracy K=5:', accuracy(du_doan_tap(x_test_s, x_train_s, y_train, 5), y_test))
ve_accuracy_theo_K(x_train_s, y_train, x_test_s, y_test)


NameError: name 'X_heart' is not defined

**Nộp kèm bài 3**

1. Có bao nhiêu người trong tập test? Tỉ lệ lớp `0` và `1` trên toàn bộ dữ liệu là bao nhiêu?
2. Accuracy cao nhất và `K` tương ứng?
3. Vì sao với bài 2 lớp nên chọn `K` lẻ?


In [ ]:
# Bài 3

# 1.
# Có 184 người trong tập test.
# Trên toàn bộ dữ liệu: lớp 0 chiếm khoảng 44.7%, lớp 1 chiếm khoảng 55.3%.

# 2.
# Accuracy cao nhất là accuracy thu được trên đồ thị accuracy theo K,
# với K tương ứng là K có accuracy cao nhất.

# 3.
# Vì bài toán có 2 lớp, chọn K lẻ giúp giảm khả năng hai lớp bị hòa phiếu
# khi các K láng giềng bỏ phiếu. Khi K lẻ, số phiếu có thể phân thắng bại
# rõ ràng hơn.

---

# Bài 4 — Mushroom Classification

Nguồn: https://www.kaggle.com/datasets/uciml/mushroom-classification

File `data/mushrooms.csv`. Đoán nấm ăn được (`e`) hay độc (`p`).

Mọi cột đều là chữ, cùng kiểu demo xe. Cột 0 là nhãn `class`. Các cột còn lại là feature.

- Bỏ dòng nào còn giá trị `?`.
- `LabelEncoder` **từng cột** feature, rồi mã hóa riêng cột nhãn. Giống đoạn mã hóa trong demo xe.
- **Không** chuẩn hóa min–max. Mã `LabelEncoder` chỉ là tên nhóm được đánh số, không phải số đo cùng đơn vị.
- Tập này vài nghìn dòng. `tinh_khoang_cach` cần tính một lần cho cả `x_train` (vector), không viết vòng `for` từng dòng train.


In [ ]:
raw_mush = np.genfromtxt('../data/mushrooms.csv', delimiter=',', dtype=str, skip_header=1)
print('trước khi bỏ dòng có ?:', raw_mush.shape)

# TODO:
mask = ~np.any(raw_mush == '?', axis=1)
mush_clean = raw_mush[mask]

# Mã hóa từng cột feature riêng
X_mush = np.zeros((mush_clean.shape[0], mush_clean.shape[1] - 1), dtype=float)

for j in range(1, mush_clean.shape[1]):
    le = LabelEncoder()
    X_mush[:, j - 1] = le.fit_transform(mush_clean[:, j])

# Mã hóa riêng cột nhãn
le_y = LabelEncoder()
Y_mush = le_y.fit_transform(mush_clean[:, 0])

print(X_mush.shape, Y_mush.shape)
print(np.unique(Y_mush, return_counts=True))


FileNotFoundError: ../data/mushrooms.csv not found.

In [13]:
x_train, y_train, x_test, y_test = chia_train_test(X_mush, Y_mush)

# Không gọi chuan_hoa_minmax
print('accuracy K=5:', accuracy(du_doan_tap(x_test, x_train, y_train, 5), y_test))
ve_accuracy_theo_K(x_train, y_train, x_test, y_test)


NameError: name 'X_mush' is not defined

**Nộp kèm bài 4**

1. Còn bao nhiêu dòng sau khi bỏ `?`? Bao nhiêu feature?
2. Accuracy cao nhất và `K` tương ứng?
3. Bài này giống demo tình trạng xe ở bước nào, và khác bài 1–3 ở chỗ nào (chuẩn hóa)?


In [ ]:
# Bài 4

# 1.
# Sau khi bỏ các dòng chứa '?', còn [điền số dòng thực tế] dòng và có 22 feature.

# 2.
# Accuracy cao nhất là [điền accuracy cao nhất] tại K = [điền K tương ứng].

# 3.
# Bài này giống demo tình trạng xe ở bước mã hóa dữ liệu:
# các cột dạng chữ được dùng LabelEncoder và mã hóa từng cột feature riêng.
# Khác bài 1–3 ở chỗ bài này KHÔNG chuẩn hóa Min-Max,
# vì các giá trị sau LabelEncoder chỉ là mã số đại diện cho các nhóm,
# không phải các đại lượng đo cùng đơn vị.